# Shortest Path Between Wikipedia Articles

This notebook demonstrates the **Wikipedia game**: find the minimum number of clicks (hyperlink hops) needed to navigate from one article to another.

The system uses **BFS** for this task because BFS guarantees it visits nodes in order of increasing distance from the start — so the first time it reaches the target, that path is definitively the shortest.

In [1]:
import sys
sys.path.insert(0, "..")

from src import BFSFrontier, WikiScraper, WikiExplorer

## Find the shortest path

In [2]:
START  = "Data science"
TARGET = "Deep learning"

explorer = WikiExplorer(
    frontier  = BFSFrontier(),
    scraper   = WikiScraper(delay=0.3),
    verbose   = True,
)

path = explorer.shortest_path(START, TARGET, max_pages=200)

[BFS shortest |    1/200] Data science
[BFS shortest |    2/200] Information science
[BFS shortest |    3/200] Computer science
[BFS shortest |    4/200] Comet NEOWISE
[BFS shortest |    5/200] Astronomical survey
[BFS shortest |    6/200] Space telescope
[BFS shortest |    7/200] Wide-field Infrared Survey Explorer
[BFS shortest |    8/200] Interdisciplinary
[BFS shortest |    9/200] Statistics
[BFS shortest |   10/200] Scientific computing
[BFS shortest |   11/200] Scientific method
[BFS shortest |   12/200] Scientific visualization
[BFS shortest |   13/200] Algorithm
[BFS shortest |   14/200] Knowledge
[BFS shortest |   15/200] Data model
[BFS shortest |   16/200] Unstructured data
[BFS shortest |   17/200] Data analysis
[BFS shortest |   18/200] Informatics
[BFS shortest |   19/200] Phenomena
[BFS shortest |   20/200] Data
[BFS shortest |   21/200] Mathematics
[BFS shortest |   22/200] Domain knowledge
[BFS shortest |   23/200] Turing Award
[BFS shortest |   24/200] Jim Gray (compu

In [3]:
if path:
    print(f"\nShortest path ({len(path) - 1} hops):")
    for i, page in enumerate(path):
        prefix = "START → " if i == 0 else ("TARGET " if i == len(path)-1 else f"  [{i}]   ")
        print(f"{prefix} {page}")
else:
    print("No path found within the page limit.")


Shortest path (2 hops):
START →  Data science
  [1]    Machine learning
TARGET  Deep learning


## Try different pairs

Edit the cell below and re-run to explore different start/target combinations.

In [4]:
PAIRS = [
    ("Data science",      "Dinosaur"),
    ("Machine learning",  "Ancient Rome"),
    ("Python (programming language)", "Shakespeare"),
]

scraper = WikiScraper(delay=0.3)

for start, target in PAIRS:
    exp = WikiExplorer(BFSFrontier(), scraper=scraper, verbose=False)
    p = exp.shortest_path(start, target, max_pages=150)
    if p:
        print(f"  {start!r:45s} → {target!r}  ({len(p)-1} hops)")
        print("  Path:", " → ".join(p))
    else:
        print(f"  {start!r} → {target!r}  NOT FOUND")
    print()

KeyboardInterrupt: 

## Visualise the path as a linear chain

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

START  = "Data science"
TARGET = "Deep learning"

exp   = WikiExplorer(BFSFrontier(), scraper=WikiScraper(delay=0.3), verbose=False)
path  = exp.shortest_path(START, TARGET, max_pages=200)

if path:
    G = nx.DiGraph()
    for i in range(len(path) - 1):
        G.add_edge(path[i], path[i + 1])

    fig, ax = plt.subplots(figsize=(max(10, len(path) * 2), 3))
    pos = {page: (i, 0) for i, page in enumerate(path)}

    colors = []
    for n in G.nodes():
        if n == START:
            colors.append("#e74c3c")
        elif n == TARGET:
            colors.append("#2ecc71")
        else:
            colors.append("#4DB8FF")

    nx.draw_networkx(
        G, pos, ax=ax,
        node_color=colors,
        node_size=1200,
        font_size=8,
        arrows=True,
        edge_color="#333",
        width=2,
    )
    start_patch  = mpatches.Patch(color="#e74c3c", label="Start")
    target_patch = mpatches.Patch(color="#2ecc71", label="Target")
    mid_patch    = mpatches.Patch(color="#4DB8FF", label="Intermediate")
    ax.legend(handles=[start_patch, mid_patch, target_patch], loc="upper left")
    ax.set_title(f"Shortest path: '{START}' → '{TARGET}'  ({len(path)-1} hops)", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("shortest_path.png", dpi=150)
    plt.show()
else:
    print("Path not found — try increasing max_pages or choosing closer topics.")

## Why BFS guarantees the shortest path

BFS explores nodes **layer by layer**. All nodes at depth *d* are fully explored before any node at depth *d+1* is touched.

```
Start
  ├── depth 1 pages  (all direct links from Start)
  │     ├── depth 2 pages
  │     │     └── ...
  │     └── ...
  └── ...
```

The moment the target is encountered, we know it's at the minimum possible depth — no shorter path can exist, because we've already exhausted all shallower layers.

DFS makes no such guarantee: it might find *a* path, but not necessarily the *shortest* one.